# Experiment 17 — Multi-Offset TTA (0, 1.25, 2.5, 3.75s)

**Building on:** exp16 single-TTA (cmAP=0.9188)

**Key idea:** Instead of averaging 2 offsets (0s + 2.5s), use 4 offsets that tile the 5s window
more evenly: 0, 1.25, 2.5, 3.75s. Should capture species calls near any boundary.

**Changes vs exp16:**
- Build 2 new Perch caches for 1.25s and 3.75s offsets
- Average all 4 predictions at logit level (equal weights)
- Same LightProtoSSM + focal+OneCycleLR + MLP probes as exp13/16

In [1]:
import os, re, gc, time, warnings, concurrent.futures
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnxruntime as ort
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

torch.manual_seed(42)
np.random.seed(42)

PROJECT_ROOT = Path('/Users/jjannik/Development/kaggle/birdclef')
BASE      = PROJECT_ROOT / 'data'
MODEL_DIR = BASE / 'models' / 'perch_tf'
ONNX_PATH = BASE / 'models' / 'perch' / 'perch_v2_no_dft.onnx'
CACHE_DIR = BASE / 'cache'

# Standard cache (exp11 — offset 0s)
CACHE_META_0   = CACHE_DIR / 'exp11_perch_meta.csv'
CACHE_NPZ_0    = CACHE_DIR / 'exp11_perch_arrays.npz'
# Existing TTA cache (exp16 — offset 2.5s)
CACHE_META_250 = CACHE_DIR / 'exp16_perch_meta_tta.csv'
CACHE_NPZ_250  = CACHE_DIR / 'exp16_perch_arrays_tta.npz'
# New caches for 1.25s and 3.75s
CACHE_META_125 = CACHE_DIR / 'exp17_perch_meta_125.csv'
CACHE_NPZ_125  = CACHE_DIR / 'exp17_perch_arrays_125.npz'
CACHE_META_375 = CACHE_DIR / 'exp17_perch_meta_375.csv'
CACHE_NPZ_375  = CACHE_DIR / 'exp17_perch_arrays_375.npz'

TTA_OFFSETS = [0.0, 1.25, 2.5, 3.75]  # seconds

SR             = 32_000
WINDOW_SEC     = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES   = 60 * SR
N_WINDOWS      = 12
N_OOF_SPLITS   = 3

CFG = dict(
    d_model=128, d_state=16, n_ssm_layers=2, dropout=0.15,
    n_sites=20, meta_dim=16,
    n_epochs=30, lr=5e-4, weight_decay=1e-3, patience=8,
    focal_gamma=1.5, distill_weight=0.15, pos_weight_cap=25.0,
    pca_dim=64, mlp_hidden=(128, 64), min_pos=5, alpha_blend=0.25,
    lambda_prior=0.10,
)

_WALL_START = time.time()
print('Setup done')

Setup done


In [2]:
# Data loading (identical to exp16)
taxonomy          = pd.read_csv(BASE / 'taxonomy.csv')
sample_sub        = pd.read_csv(BASE / 'sample_submission.csv')
soundscape_labels = pd.read_csv(BASE / 'train_soundscapes_labels.csv')

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}
CLASS_NAME_MAP = taxonomy.set_index('primary_label')['class_name'].to_dict()

FNAME_RE = re.compile(r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg')

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m: return {'site': 'unknown', 'hour_utc': -1}
    _, site, _, hms = m.groups()
    return {'site': site, 'hour_utc': int(hms[:2])}

def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(';'):
                t = t.strip()
                if t: out.add(t)
    return sorted(out)

sc = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc['end_sec'] = pd.to_timedelta(sc['end']).dt.total_seconds().astype(int)
sc['row_id']  = sc['filename'].str.replace('.ogg', '', regex=False) + '_' + sc['end_sec'].astype(str)
_meta = sc['filename'].apply(parse_fname).apply(pd.Series)
sc = pd.concat([sc, _meta], axis=1)

Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, lbls in enumerate(sc['label_list']):
    for lbl in lbls:
        if lbl in label_to_idx: Y_SC[i, label_to_idx[lbl]] = 1

windows_per_file = sc.groupby('filename').size()
LABEL_WINDOWS    = int(windows_per_file.mode().iloc[0])
full_files       = sorted(windows_per_file[windows_per_file == LABEL_WINDOWS].index.tolist())
sc['fully_labeled'] = sc['filename'].isin(full_files)
full_rows = sc[sc['fully_labeled']].sort_values(['filename', 'end_sec']).reset_index(drop=False)

print(f'Classes: {N_CLASSES} | Fully-labeled files: {len(full_files)}')

Classes: 234 | Fully-labeled files: 59


In [3]:
# Load Perch ONNX + label mapping
import re as _re

_so = ort.SessionOptions(); _so.intra_op_num_threads = 4
ONNX_SESSION    = ort.InferenceSession(str(ONNX_PATH), sess_options=_so, providers=['CPUExecutionProvider'])
ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
ONNX_OUT_MAP    = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}

bc_labels = (pd.read_csv(MODEL_DIR / 'assets' / 'labels.csv')
             .reset_index().rename(columns={'index': 'bc_index', 'inat2024_fsd50k': 'scientific_name'}))
NO_LABEL  = len(bc_labels)
mapping   = taxonomy.merge(bc_labels, on='scientific_name', how='left')
mapping['bc_index'] = mapping['bc_index'].fillna(NO_LABEL).astype(int)
lbl2bc    = mapping.set_index('primary_label')['bc_index']
BC_INDICES    = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK   = BC_INDICES != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)

UNMAPPED_POS = np.where(~MAPPED_MASK)[0].astype(np.int32)
proxy_map    = {}
for _, row in taxonomy[taxonomy['primary_label'].isin([PRIMARY_LABELS[i] for i in UNMAPPED_POS])].iterrows():
    genus = str(row['scientific_name']).split()[0]
    hits  = bc_labels[bc_labels['scientific_name'].astype(str).str.match(rf'^{_re.escape(genus)}\s', na=False)]
    if len(hits) > 0: proxy_map[label_to_idx[row['primary_label']]] = hits['bc_index'].astype(int).tolist()
proxy_map = {k: v for k, v in proxy_map.items() if CLASS_NAME_MAP.get(PRIMARY_LABELS[k]) in {'Amphibia','Insecta','Aves'}}

print(f'Perch loaded. Mapped: {MAPPED_MASK.sum()}/{N_CLASSES}')

Perch loaded. Mapped: 203/234

In [4]:
# Perch inference engine for arbitrary offset

def read_60s_shifted(path, offset_sec):
    y, sr = sf.read(path, dtype='float32', always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    offset_samples = int(offset_sec * SR)
    needed = offset_samples + N_WINDOWS * WINDOW_SAMPLES
    if len(y) < needed: y = np.pad(y, (0, needed - len(y)))
    return y[offset_samples : offset_samples + N_WINDOWS * WINDOW_SAMPLES]

def run_perch_shifted(paths, offset_sec, batch_files=8, verbose=True):
    paths  = [Path(p) for p in paths]
    n_rows = len(paths) * N_WINDOWS
    row_ids   = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites     = np.empty(n_rows, dtype=object)
    hours     = np.zeros(n_rows, dtype=np.int16)
    scores    = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embs      = np.zeros((n_rows, 1536), dtype=np.float32)

    wr = 0
    itr = tqdm(range(0, len(paths), batch_files), desc=f'Perch {offset_sec}s') if verbose else range(0, len(paths), batch_files)

    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as exe:
        nxt_p = paths[:batch_files]
        nxt_f = [exe.submit(read_60s_shifted, p, offset_sec) for p in nxt_p]
        for start in itr:
            bp = nxt_p; ba = [f.result() for f in nxt_f]; br = wr
            ns = start + batch_files
            if ns < len(paths):
                nxt_p = paths[ns:ns+batch_files]
                nxt_f = [exe.submit(read_60s_shifted, p, offset_sec) for p in nxt_p]
            x = np.stack(ba).reshape(len(bp) * N_WINDOWS, WINDOW_SAMPLES)
            for bi, path in enumerate(bp):
                meta = parse_fname(path.name); stem = path.stem
                row_ids[wr:wr+N_WINDOWS]   = [f'{stem}_off{offset_sec}_{i}' for i in range(N_WINDOWS)]
                filenames[wr:wr+N_WINDOWS] = path.name
                sites[wr:wr+N_WINDOWS]     = meta['site']
                hours[wr:wr+N_WINDOWS]     = meta['hour_utc']
                wr += N_WINDOWS
            outs   = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
            logits = outs[ONNX_OUT_MAP['label']].astype(np.float32)
            emb    = outs[ONNX_OUT_MAP['embedding']].astype(np.float32)
            scores[br:wr, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
            embs  [br:wr]             = emb
            for pos_idx, bc_idxs in proxy_map.items():
                scores[br:wr, pos_idx] = logits[:, np.array(bc_idxs)].max(axis=1)
            del x, outs, logits, emb, ba; gc.collect()

    meta_df = pd.DataFrame({'row_id': row_ids, 'filename': filenames, 'site': sites, 'hour_utc': hours})
    return meta_df, scores, embs

print('Perch engine ready')

Perch engine ready


In [5]:
# Load all 4 TTA caches (building missing ones)

def _pick_array(arr, ncols):
    for k in arr.files:
        v = arr[k]
        if v.ndim == 2 and v.shape[1] == ncols: return v
    raise KeyError('array not found')

train_paths = [BASE / 'train_soundscapes' / fn for fn in full_files]

# offset 0.0 — standard cache from exp11
print('Loading offset 0.0s (exp11 cache)...')
meta_0 = pd.read_csv(CACHE_META_0)
_a     = np.load(CACHE_NPZ_0)
sc_0   = _pick_array(_a, N_CLASSES).astype(np.float32)
emb_0  = _pick_array(_a, 1536).astype(np.float32)
row_id_to_index = full_rows.set_index('row_id')['index']
Y_FULL_aligned  = Y_SC[row_id_to_index.loc[meta_0['row_id']].to_numpy()]
print(f'  sc_0={sc_0.shape} emb_0={emb_0.shape}')

# offset 2.5 — from exp16
print('Loading offset 2.5s (exp16 cache)...')
meta_250 = pd.read_csv(CACHE_META_250)
_a       = np.load(CACHE_NPZ_250)
sc_250   = _pick_array(_a, N_CLASSES).astype(np.float32)
emb_250  = _pick_array(_a, 1536).astype(np.float32)
print(f'  sc_250={sc_250.shape}')

# offset 1.25
if CACHE_META_125.exists() and CACHE_NPZ_125.exists():
    print('Loading offset 1.25s...')
    meta_125 = pd.read_csv(CACHE_META_125)
    _a       = np.load(CACHE_NPZ_125)
    sc_125   = _pick_array(_a, N_CLASSES).astype(np.float32)
    emb_125  = _pick_array(_a, 1536).astype(np.float32)
else:
    print('Building offset 1.25s cache...')
    t0 = time.time()
    meta_125, sc_125, emb_125 = run_perch_shifted(train_paths, 1.25)
    print(f'  Done in {time.time()-t0:.1f}s')
    meta_125.to_csv(CACHE_META_125, index=False)
    np.savez(CACHE_NPZ_125, scores=sc_125.astype(np.float32), embs=emb_125.astype(np.float32))
print(f'  sc_125={sc_125.shape}')

# offset 3.75
if CACHE_META_375.exists() and CACHE_NPZ_375.exists():
    print('Loading offset 3.75s...')
    meta_375 = pd.read_csv(CACHE_META_375)
    _a       = np.load(CACHE_NPZ_375)
    sc_375   = _pick_array(_a, N_CLASSES).astype(np.float32)
    emb_375  = _pick_array(_a, 1536).astype(np.float32)
else:
    print('Building offset 3.75s cache...')
    t0 = time.time()
    meta_375, sc_375, emb_375 = run_perch_shifted(train_paths, 3.75)
    print(f'  Done in {time.time()-t0:.1f}s')
    meta_375.to_csv(CACHE_META_375, index=False)
    np.savez(CACHE_NPZ_375, scores=sc_375.astype(np.float32), embs=emb_375.astype(np.float32))
print(f'  sc_375={sc_375.shape}')

meta_tr = meta_0  # reference metadata
print('All caches loaded.')

Loading offset 0.0s (exp11 cache)...
  sc_0=(708, 234) emb_0=(708, 1536)
Loading offset 2.5s (exp16 cache)...
  sc_250=(708, 234)
Building offset 1.25s cache...


Perch 1.25s:   0%|          | 0/8 [00:00<?, ?it/s]

  Done in 16.3s
  sc_125=(708, 234)
Building offset 3.75s cache...


Perch 3.75s:   0%|          | 0/8 [00:00<?, ?it/s]

  Done in 16.0s
  sc_375=(708, 234)
All caches loaded.


In [6]:
# Metrics + loss (identical to exp16)
def macro_auc(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average='macro')

def padded_cmap(y_true, y_pred, pad=5):
    aps = []
    for c in range(y_true.shape[1]):
        yt = np.concatenate([y_true[:, c], np.ones(pad)])
        yp = np.concatenate([y_pred[:, c], np.ones(pad)])
        aps.append(average_precision_score(yt, yp))
    return float(np.mean(aps))

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def focal_bce(logits, targets, gamma=1.5, pos_weight=None):
    bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight, reduction='none')
    p_t = torch.exp(-bce)
    return ((1 - p_t) ** gamma * bce).mean()

print('Metrics ready')

Metrics ready


In [7]:
# Post-processing helpers (identical to exp16)

def build_prior_tables(sc_df, Y_labels):
    sc_df = sc_df.reset_index(drop=True)
    global_p = Y_labels.mean(axis=0).astype(np.float32)
    site_keys = sorted(sc_df['site'].dropna().astype(str).unique())
    site_to_i = {k: i for i, k in enumerate(site_keys)}
    site_p    = np.zeros((len(site_keys), Y_labels.shape[1]), dtype=np.float32)
    site_n    = np.zeros(len(site_keys), dtype=np.float32)
    for s in site_keys:
        i = site_to_i[s]; mask = sc_df['site'].astype(str).values == s
        site_n[i] = mask.sum(); site_p[i] = Y_labels[mask].mean(axis=0)
    hour_keys = sorted(sc_df['hour_utc'].dropna().astype(int).unique())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_p    = np.zeros((len(hour_keys), Y_labels.shape[1]), dtype=np.float32)
    hour_n    = np.zeros(len(hour_keys), dtype=np.float32)
    for h in hour_keys:
        i = hour_to_i[h]; mask = sc_df['hour_utc'].astype(int).values == h
        hour_n[i] = mask.sum(); hour_p[i] = Y_labels[mask].mean(axis=0)
    return {'global_p': global_p,
            'site_to_i': site_to_i, 'site_p': site_p, 'site_n': site_n,
            'hour_to_i': hour_to_i, 'hour_p': hour_p, 'hour_n': hour_n}

def apply_prior(scores, sites, hours, tables, lambda_prior=0.10):
    eps = 1e-4; out = scores.copy()
    p   = np.tile(tables['global_p'], (len(scores), 1))
    for i, h in enumerate(hours):
        h = int(h)
        if h in tables['hour_to_i']:
            j = tables['hour_to_i'][h]; nh = tables['hour_n'][j]
            w = nh / (nh + 8.0)
            p[i] = w * tables['hour_p'][j] + (1 - w) * tables['global_p']
    for i, s in enumerate(sites):
        s = str(s)
        if s in tables['site_to_i']:
            j = tables['site_to_i'][s]; ns = tables['site_n'][j]
            w = ns / (ns + 8.0)
            p[i] = w * tables['site_p'][j] + (1 - w) * p[i]
    p = np.clip(p, eps, 1 - eps)
    out += lambda_prior * (np.log(p) - np.log1p(-p))
    return out.astype(np.float32)

def adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=0.10):
    result = probs.copy(); view = probs.reshape(-1, n_windows, probs.shape[1])
    out    = result.reshape(-1, n_windows, probs.shape[1])
    for t in range(n_windows):
        conf = view[:, t, :].max(axis=-1, keepdims=True)
        alpha = base_alpha * (1.0 - conf)
        nbr = (view[:, t-1, :] + view[:, t+1, :]) / 2.0 if 0 < t < n_windows-1 else \
              (view[:, t, :] + view[:, t+1, :]) / 2.0 if t == 0 else \
              (view[:, t-1, :] + view[:, t, :]) / 2.0
        out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * nbr
    return result

def file_confidence_scale(probs, n_windows=N_WINDOWS, top_k=2, power=0.05):
    view = probs.reshape(-1, n_windows, probs.shape[1])
    top_mean = np.sort(view, axis=1)[:, -top_k:, :].mean(axis=1, keepdims=True)
    return (view * np.power(top_mean, power)).reshape(probs.shape)

TEXTURE_TAXA = {'Amphibia', 'Insecta'}
temperatures = np.ones(N_CLASSES, dtype=np.float32)
for ci, lbl in enumerate(PRIMARY_LABELS):
    temperatures[ci] = 0.95 if CLASS_NAME_MAP.get(lbl, 'Aves') in TEXTURE_TAXA else 1.10

print('Helpers ready')

Helpers ready


In [8]:
# MLP probes (identical to exp16)

def build_class_freq_weights(Y, cap=10.0):
    pos_count = Y.sum(axis=0).astype(np.float32) + 1.0
    freq      = pos_count / Y.shape[0]
    weights   = np.clip(1.0 / (freq ** 0.5), 1.0, cap)
    return (weights / weights.mean()).astype(np.float32)

def build_seq_features(scores_col, n_windows=N_WINDOWS):
    x     = scores_col.reshape(-1, n_windows)
    prev  = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    next_ = np.concatenate([x[:, 1:], x[:, -1:]],  axis=1).reshape(-1)
    mean  = np.repeat(x.mean(axis=1), n_windows)
    max_  = np.repeat(x.max(axis=1),  n_windows)
    std   = np.repeat(x.std(axis=1),  n_windows)
    return prev, next_, mean, max_, std

def train_mlp_probes(emb, scores_raw, Y, pca_dim=64, alpha_blend=0.25, min_pos=5):
    scaler = StandardScaler()
    emb_s  = scaler.fit_transform(emb)
    pca    = PCA(n_components=min(pca_dim, emb_s.shape[1] - 1))
    Z      = pca.fit_transform(emb_s).astype(np.float32)
    print(f'  PCA: {emb.shape} → {Z.shape}  ({pca.explained_variance_ratio_.sum():.2%})')
    class_weights = build_class_freq_weights(Y, cap=10.0)
    probe_models  = {}
    active = np.where(Y.sum(axis=0) >= min_pos)[0]
    print(f'  Training MLP probes for {len(active)} species...')
    MAX_ROWS = 2000
    for ci in active:
        y = Y[:, ci]
        if y.sum() == 0 or y.sum() == len(y): continue
        prev, next_, mean, max_, std = build_seq_features(scores_raw[:, ci])
        X = np.hstack([Z, scores_raw[:, ci:ci+1],
                       prev[:, None], next_[:, None],
                       mean[:, None], max_[:, None], std[:, None]])
        n_pos  = int(y.sum()); pos_idx = np.where(y == 1)[0]
        w      = float(class_weights[ci])
        repeat = min(max(1, int(round(w * (len(y) - n_pos) / max(n_pos, 1)))), 8)
        if n_pos * repeat + len(y) > MAX_ROWS:
            repeat = max(1, (MAX_ROWS - len(y)) // max(n_pos, 1))
        X_bal = np.vstack([X, np.tile(X[pos_idx], (repeat, 1))])
        y_bal = np.concatenate([y, np.ones(n_pos * repeat, dtype=y.dtype)])
        clf = MLPClassifier(
            hidden_layer_sizes=CFG['mlp_hidden'], activation='relu',
            max_iter=200, early_stopping=True,
            validation_fraction=0.15, n_iter_no_change=15,
            random_state=42, learning_rate_init=5e-4, alpha=0.005,
        )
        clf.fit(X_bal, y_bal)
        probe_models[ci] = clf
    print(f'  Trained {len(probe_models)} probes')
    return probe_models, scaler, pca, alpha_blend

def apply_mlp_probes(emb_test, scores_test, probe_models, scaler, pca, alpha_blend=0.25):
    Z_test = pca.transform(scaler.transform(emb_test)).astype(np.float32)
    result = scores_test.copy()
    for ci, clf in probe_models.items():
        prev, next_, mean, max_, std = build_seq_features(scores_test[:, ci])
        X_test = np.hstack([Z_test, scores_test[:, ci:ci+1],
                            prev[:, None], next_[:, None],
                            mean[:, None], max_[:, None], std[:, None]])
        prob  = clf.predict_proba(X_test)[:, 1].astype(np.float32)
        logit = np.log(prob + 1e-7) - np.log(1 - prob + 1e-7)
        result[:, ci] = (1 - alpha_blend) * scores_test[:, ci] + alpha_blend * logit
    return result

print('MLP probes ready')

MLP probes ready


In [9]:
# LightProtoSSM (identical to exp16)

class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model; self.d_state = d_state
        self.in_proj  = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d   = nn.Conv1d(d_model, d_model, d_conv, padding=d_conv-1, groups=d_model)
        self.dt_proj  = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state+1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A)); self.D = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)

    def forward(self, x):
        B_sz, T, D = x.shape
        xz = self.in_proj(x); x_ssm, _ = xz.chunk(2, dim=-1)
        x_c = self.conv1d(x_ssm.transpose(1,2))[:,:,:T].transpose(1,2)
        x_c = F.silu(x_c)
        dt  = F.softplus(self.dt_proj(x_c))
        A   = -torch.exp(self.A_log)
        B_  = self.B_proj(x_c); C = self.C_proj(x_c)
        h   = torch.zeros(B_sz, D, self.d_state, device=x.device)
        ys  = []
        for t in range(T):
            h = h * torch.exp(A[None] * dt[:,t,:,None]) + x[:,t,:,None] * (dt[:,t,:,None] * B_[:,t,None,:])
            ys.append((h * C[:,t,None,:]).sum(-1))
        return torch.stack(ys, dim=1) + x * self.D[None, None, :]


class LightProtoSSM(nn.Module):
    def __init__(self, d_input=1536, d_model=128, d_state=16, n_ssm_layers=2,
                 n_classes=234, n_windows=N_WINDOWS, dropout=0.15,
                 n_sites=20, meta_dim=16, cross_attn_heads=2):
        super().__init__()
        self.n_classes = n_classes; self.n_windows = n_windows
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_enc   = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_emb  = nn.Embedding(n_sites, meta_dim)
        self.hour_emb  = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.ssm_fwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(n_ssm_layers)])
        self.ssm_bwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(n_ssm_layers)])
        self.ssm_merge = nn.ModuleList([nn.Linear(2 * d_model, d_model) for _ in range(n_ssm_layers)])
        self.ssm_norm  = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_ssm_layers)])
        self.drop      = nn.Dropout(dropout)
        self.cross_attn = nn.ModuleList([
            nn.MultiheadAttention(d_model, num_heads=cross_attn_heads, dropout=dropout, batch_first=True)
            for _ in range(n_ssm_layers)])
        self.cross_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_ssm_layers)])
        self.prototypes  = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp  = nn.Parameter(torch.tensor(5.0))
        self.class_bias  = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))

    def init_prototypes(self, emb_tensor, labels_tensor):
        with torch.no_grad():
            h = self.input_proj(emb_tensor)
            for c in range(self.n_classes):
                mask = labels_tensor[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(emb) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat([
                self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings-1)),
                self.hour_emb(hours.clamp(0, 23))], dim=-1))
            h = h + meta[:, None, :]
        for fwd, bwd, merge, norm, ca, cn in zip(
                self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm,
                self.cross_attn, self.cross_norm):
            res = h
            h_f = fwd(h); h_b = bwd(h.flip(1)).flip(1)
            h   = self.drop(merge(torch.cat([h_f, h_b], dim=-1)))
            h   = norm(h + res)
            attn_out, _ = ca(h, h, h); h = cn(h + attn_out)
        h_n = F.normalize(h, dim=-1); p_n = F.normalize(self.prototypes, dim=-1)
        sim = torch.matmul(h_n, p_n.T) * F.softplus(self.proto_temp) + self.class_bias[None,None,:]
        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
            return alpha * sim + (1.0 - alpha) * perch_logits
        return sim


def train_proto_ssm(emb_full, scores_full, Y_full, meta_full, n_epochs=30, patience=8, lr=5e-4, n_sites=20):
    n_files = len(emb_full) // N_WINDOWS
    emb_f = emb_full.reshape(n_files, N_WINDOWS, -1)
    log_f = scores_full.reshape(n_files, N_WINDOWS, -1)
    lab_f = Y_full.reshape(n_files, N_WINDOWS, -1).astype(np.float32)
    fnames  = meta_full.groupby('filename', sort=False).size().index.tolist()
    sites_u = sorted(meta_full['site'].astype(str).unique())
    site2i  = {s: i+1 for i, s in enumerate(sites_u)}
    site_ids = np.array([min(site2i.get(str(meta_full.loc[meta_full['filename']==fn,'site'].iloc[0]), 0), n_sites-1) for fn in fnames], dtype=np.int64)
    hour_ids = np.array([int(meta_full.loc[meta_full['filename']==fn,'hour_utc'].iloc[0]) % 24 for fn in fnames], dtype=np.int64)
    model = LightProtoSSM(d_model=CFG['d_model'], d_state=CFG['d_state'],
                          n_ssm_layers=CFG['n_ssm_layers'], n_classes=N_CLASSES,
                          n_windows=N_WINDOWS, dropout=CFG['dropout'],
                          n_sites=n_sites, meta_dim=CFG['meta_dim'])
    model.init_prototypes(torch.tensor(emb_full, dtype=torch.float32), torch.tensor(Y_full, dtype=torch.float32))
    emb_t = torch.tensor(emb_f, dtype=torch.float32)
    log_t = torch.tensor(log_f, dtype=torch.float32)
    lab_t = torch.tensor(lab_f, dtype=torch.float32)
    site_t = torch.tensor(site_ids, dtype=torch.long)
    hour_t = torch.tensor(hour_ids, dtype=torch.long)
    pos_cnt = lab_t.sum(dim=(0,1))
    pos_weight = ((lab_t.shape[0]*lab_t.shape[1] - pos_cnt) / (pos_cnt+1)).clamp(max=CFG['pos_weight_cap'])
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG['weight_decay'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=1, pct_start=0.3, anneal_strategy='cos')
    best_loss = float('inf'); best_state = None; wait = 0
    for ep in range(n_epochs):
        model.train()
        out  = model(emb_t, log_t, site_ids=site_t, hours=hour_t)
        loss = (focal_bce(out, lab_t, gamma=CFG['focal_gamma'], pos_weight=pos_weight[None,None,:])
                + CFG['distill_weight'] * F.mse_loss(out, log_t))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}; wait = 0
        else:
            wait += 1
        if wait >= patience: break
    if best_state: model.load_state_dict(best_state)
    model.eval(); return model, site2i

print('LightProtoSSM ready')

LightProtoSSM ready


In [10]:
# Full OOF pipeline with 4-offset TTA
prior_tables = build_prior_tables(sc, Y_SC)
file_meta    = meta_tr.drop_duplicates('filename').reset_index(drop=True)
gkf          = GroupKFold(n_splits=N_OOF_SPLITS)

oof_probs_std  = np.zeros((len(sc_0), N_CLASSES), dtype=np.float32)  # 0s only
oof_probs_tta4 = np.zeros((len(sc_0), N_CLASSES), dtype=np.float32)  # 4-offset average
fold_aucs_std  = []; fold_aucs_tta4 = []

for fold, (tr_f, va_f) in enumerate(
        gkf.split(file_meta, groups=file_meta['filename']), 1):
    t_fold = time.time()
    tr_fnames = set(file_meta.iloc[tr_f]['filename'])
    va_fnames = set(file_meta.iloc[va_f]['filename'])
    tr_mask   = meta_tr['filename'].isin(tr_fnames).values
    va_mask   = meta_tr['filename'].isin(va_fnames).values

    print(f'\nFold {fold}/{N_OOF_SPLITS} — train: {len(tr_fnames)}, val: {len(va_fnames)}')

    # --- Train ProtoSSM on standard windows ---
    proto_model, site2i = train_proto_ssm(
        emb_0[tr_mask], sc_0[tr_mask], Y_FULL_aligned[tr_mask],
        meta_tr[tr_mask].reset_index(drop=True))

    # --- Train MLP probes on standard windows ---
    probe_models, emb_scaler, emb_pca, alpha_blend = train_mlp_probes(
        emb_0[tr_mask], sc_0[tr_mask], Y_FULL_aligned[tr_mask],
        pca_dim=CFG['pca_dim'])

    n_va       = int(va_mask.sum()) // N_WINDOWS
    va_fn_list = meta_tr[va_mask].drop_duplicates('filename')['filename'].tolist()
    va_site_ids = np.array([min(site2i.get(str(meta_tr.loc[meta_tr['filename']==fn,'site'].iloc[0]), 0), 19) for fn in va_fn_list], dtype=np.int64)
    va_hour_ids = np.array([int(meta_tr.loc[meta_tr['filename']==fn,'hour_utc'].iloc[0]) % 24 for fn in va_fn_list], dtype=np.int64)
    site_t = torch.tensor(va_site_ids, dtype=torch.long)
    hour_t = torch.tensor(va_hour_ids, dtype=torch.long)

    def _infer_offset(sc_off, emb_off):
        """Get blended logits for one TTA offset."""
        proto_model.eval()
        with torch.no_grad():
            proto_out = proto_model(
                torch.tensor(emb_off[va_mask].reshape(n_va, N_WINDOWS, -1), dtype=torch.float32),
                torch.tensor(sc_off[va_mask].reshape(n_va, N_WINDOWS, -1), dtype=torch.float32),
                site_ids=site_t, hours=hour_t,
            ).numpy().reshape(-1, N_CLASSES)
        mlp_out  = apply_mlp_probes(emb_off[va_mask], sc_off[va_mask], probe_models, emb_scaler, emb_pca, alpha_blend)
        prior_out = apply_prior(sc_off[va_mask],
                                sites=meta_tr[va_mask]['site'].to_numpy(),
                                hours=meta_tr[va_mask]['hour_utc'].to_numpy(),
                                tables=prior_tables, lambda_prior=CFG['lambda_prior'])
        return 0.20 * proto_out + 0.55 * mlp_out + 0.25 * prior_out

    # Infer all 4 offsets
    fp_0   = _infer_offset(sc_0,   emb_0)
    fp_125 = _infer_offset(sc_125, emb_125)
    fp_250 = _infer_offset(sc_250, emb_250)
    fp_375 = _infer_offset(sc_375, emb_375)

    # Standard (0s only)
    def _postprocess(logits):
        p = sigmoid(logits / temperatures[None, :])
        p = file_confidence_scale(p, power=0.05)
        p = adaptive_delta_smooth(p, base_alpha=0.10)
        return np.clip(p, 0.0, 1.0)

    probs_std  = _postprocess(fp_0)
    probs_tta4 = _postprocess(0.25 * fp_0 + 0.25 * fp_125 + 0.25 * fp_250 + 0.25 * fp_375)

    Y_va = Y_FULL_aligned[va_mask]
    auc_std  = macro_auc(Y_va, probs_std)
    auc_tta4 = macro_auc(Y_va, probs_tta4)
    fold_aucs_std.append(auc_std)
    fold_aucs_tta4.append(auc_tta4)
    oof_probs_std[va_mask]  = probs_std
    oof_probs_tta4[va_mask] = probs_tta4

    print(f'  Fold {fold}: std AUC={auc_std:.5f}  4-TTA AUC={auc_tta4:.5f}  ({time.time()-t_fold:.0f}s)')

print(f'\nStd   AUC: {np.mean(fold_aucs_std):.5f}')
print(f'4-TTA AUC: {np.mean(fold_aucs_tta4):.5f}')


Fold 1/3 — train: 39, val: 20


  PCA: (468, 1536) → (468, 64)  (84.62%)
  Training MLP probes for 45 species...


  Trained 45 probes
  Fold 1: std AUC=0.92813  4-TTA AUC=0.92974  (21s)

Fold 2/3 — train: 39, val: 20


  PCA: (468, 1536) → (468, 64)  (83.72%)
  Training MLP probes for 53 species...


  Trained 53 probes
  Fold 2: std AUC=0.92609  4-TTA AUC=0.91872  (21s)

Fold 3/3 — train: 40, val: 19


  PCA: (480, 1536) → (480, 64)  (84.51%)
  Training MLP probes for 51 species...


  Trained 51 probes
  Fold 3: std AUC=0.90743  4-TTA AUC=0.90277  (21s)

Std   AUC: 0.92055
4-TTA AUC: 0.91708


In [11]:
# Final results
oof_auc_std   = macro_auc(Y_FULL_aligned, oof_probs_std)
oof_cmap_std  = padded_cmap(Y_FULL_aligned, oof_probs_std)

oof_auc_tta4  = macro_auc(Y_FULL_aligned, oof_probs_tta4)
oof_cmap_tta4 = padded_cmap(Y_FULL_aligned, oof_probs_tta4)

raw_probs = sigmoid(sc_0 / temperatures[None, :])
raw_auc   = macro_auc(Y_FULL_aligned, raw_probs)
raw_cmap  = padded_cmap(Y_FULL_aligned, raw_probs)

print('=' * 60)
print(f'Raw Perch  OOF AUC={raw_auc:.5f}  cmAP={raw_cmap:.5f}')
print(f'Std (0s)   OOF AUC={oof_auc_std:.5f}  cmAP={oof_cmap_std:.5f}')
print(f'4-TTA      OOF AUC={oof_auc_tta4:.5f}  cmAP={oof_cmap_tta4:.5f}')
print(f'4-TTA delta:   AUC={oof_auc_tta4 - oof_auc_std:+.5f}  cmAP={oof_cmap_tta4 - oof_cmap_std:+.5f}')
print(f'vs exp16 (0.9188): delta cmAP = {oof_cmap_tta4 - 0.9188:+.5f}')
print(f'vs exp11 (0.9166): delta cmAP = {oof_cmap_tta4 - 0.9166:+.5f}')
print(f'Total wall time: {(time.time() - _WALL_START)/60:.1f} min')

Raw Perch  OOF AUC=0.73902  cmAP=0.86916
Std (0s)   OOF AUC=0.92381  cmAP=0.91837
4-TTA      OOF AUC=0.92394  cmAP=0.91772
4-TTA delta:   AUC=+0.00013  cmAP=-0.00064
vs exp16 (0.9188): delta cmAP = -0.00108
vs exp11 (0.9166): delta cmAP = +0.00112
Total wall time: 1.6 min
